# Combine Monthly Data
## Reddit r/politics — 2024 US Presidential Election

**Pipeline position:** **Combine_Monthly_Data.ipynb** → Data_preparation.ipynb → VADER_Labeling.ipynb → EDA.ipynb

For each month, combines the base JSONL file with the month-end supplement file.
Deduplicates by ID and saves a single combined JSONL per month for both posts and comments.

## 0. Setup

In [5]:
import json
from pathlib import Path

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'Datasources'

print('DATA_DIR:', DATA_DIR)
print('Files found:', len(list(DATA_DIR.glob('*.jsonl'))))


DATA_DIR: C:\Users\Dell\Desktop\NLP\Final project\Datasources
Files found: 24


## 1. Define File Pairs

Each month has a base file + a month-end supplement. November has no supplement yet.
Output files are saved as `r_politics_posts_<Month>_combined.jsonl` etc.

In [6]:
# (base_file, supplement_file_or_None, output_file)
POST_PAIRS = [
    ('r_politics_posts_June.jsonl',      'r_politics_posts_June30.jsonl',   'r_politics_posts_June_combined.jsonl'),
    ('r_politics_posts_July.jsonl',      'r_politics_posts_July31.jsonl',   'r_politics_posts_July_combined.jsonl'),
    ('r_politics_posts_August.jsonl',    'r_politics_posts_August31.jsonl', 'r_politics_posts_August_combined.jsonl'),
    ('r_politics_posts_September.jsonl', 'r_politics_posts_Sept30.jsonl',   'r_politics_posts_September_combined.jsonl'),
    ('r_politics_posts_Oct.jsonl',       'r_politics_posts_Oct31.jsonl',    'r_politics_posts_October_combined.jsonl'),
    ('r_politics_posts_November.jsonl',  'r_politics_posts_November6.jsonl', 'r_politics_posts_November_combined.jsonl'),
]

COMMENT_PAIRS = [
    ('r_politics_comments_June.jsonl',      'r_politics_comments_June30.jsonl',      'r_politics_comments_June_combined.jsonl'),
    ('r_politics_comments_July.jsonl',      'r_politics_comments_July31.jsonl',      'r_politics_comments_July_combined.jsonl'),
    ('r_politics_comments_August.jsonl',    'r_politics_comments_August31.jsonl',    'r_politics_comments_August_combined.jsonl'),
    ('r_politics_comments_September.jsonl', 'r_politics_comments_September30.jsonl', 'r_politics_comments_September_combined.jsonl'),
    ('r_politics_comments_Oct.jsonl',       'r_politics_comments_Oct31.jsonl',       'r_politics_comments_October_combined.jsonl'),
    ('r_politics_comments_November.jsonl',  'r_politics_comments_November6.jsonl',   'r_politics_comments_November_combined.jsonl'),
]


## 2. Combine Posts

Reads each pair, deduplicates by `id`, writes combined JSONL.

In [7]:
def combine_jsonl(base_file, supplement_file, output_file, id_field='id'):
    seen_ids = set()
    records  = []

    for fpath in [base_file, supplement_file]:
        if fpath is None or not fpath.exists():
            continue
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    rid = obj.get(id_field)
                    if rid and rid not in seen_ids:
                        seen_ids.add(rid)
                        records.append(line)
                except json.JSONDecodeError:
                    pass

    with open(output_file, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(rec + '\n')

    return len(records)

print('=== Combining Posts ===')
for base, supp, out in POST_PAIRS:
    base_path = DATA_DIR / base
    supp_path = DATA_DIR / supp if supp else None
    out_path  = DATA_DIR / out

    if not base_path.exists():
        print(f'  SKIP {base} (not found)')
        continue

    base_lines = sum(1 for _ in open(base_path, encoding='utf-8', errors='replace'))
    supp_lines = sum(1 for _ in open(supp_path, encoding='utf-8', errors='replace')) if supp_path and supp_path.exists() else 0
    n = combine_jsonl(base_path, supp_path, out_path)
    dupes = base_lines + supp_lines - n
    print(f'  {out}')
    print(f'    base: {base_lines:,}  +  supplement: {supp_lines:,}  ->  combined: {n:,}  (dupes removed: {dupes:,})')


=== Combining Posts ===
  r_politics_posts_June_combined.jsonl
    base: 9,440  +  supplement: 380  ->  combined: 9,820  (dupes removed: 0)
  r_politics_posts_July_combined.jsonl
    base: 20,082  +  supplement: 734  ->  combined: 20,816  (dupes removed: 0)
  r_politics_posts_August_combined.jsonl
    base: 17,425  +  supplement: 339  ->  combined: 17,764  (dupes removed: 0)
  r_politics_posts_September_combined.jsonl
    base: 14,204  +  supplement: 392  ->  combined: 14,596  (dupes removed: 0)
  r_politics_posts_October_combined.jsonl
    base: 15,423  +  supplement: 760  ->  combined: 16,183  (dupes removed: 0)
  r_politics_posts_November_combined.jsonl
    base: 3,790  +  supplement: 2,772  ->  combined: 6,562  (dupes removed: 0)


## 3. Combine Comments - Do not Rerun!!!

Same process for comments — deduplicates by `id`.

In [8]:
print('=== Combining Comments ===')
for base, supp, out in COMMENT_PAIRS:
    base_path = DATA_DIR / base
    supp_path = DATA_DIR / supp if supp else None
    out_path  = DATA_DIR / out

    if not base_path.exists():
        print(f'  SKIP {base} (not found)')
        continue

    base_lines = sum(1 for _ in open(base_path, encoding='utf-8', errors='replace'))
    supp_lines = sum(1 for _ in open(supp_path, encoding='utf-8', errors='replace')) if supp_path and supp_path.exists() else 0
    n = combine_jsonl(base_path, supp_path, out_path)
    dupes = base_lines + supp_lines - n
    print(f'  {out}')
    print(f'    base: {base_lines:,}  +  supplement: {supp_lines:,}  ->  combined: {n:,}  (dupes removed: {dupes:,})')


=== Combining Comments ===
  r_politics_comments_June_combined.jsonl
    base: 760,533  +  supplement: 46,050  ->  combined: 806,583  (dupes removed: 0)
  r_politics_comments_July_combined.jsonl
    base: 2,000,001  +  supplement: 1,000  ->  combined: 2,001,001  (dupes removed: 0)
  r_politics_comments_August_combined.jsonl
    base: 1,431,008  +  supplement: 26,897  ->  combined: 1,457,905  (dupes removed: 0)
  r_politics_comments_September_combined.jsonl
    base: 1,160,860  +  supplement: 28,270  ->  combined: 1,189,130  (dupes removed: 0)
  r_politics_comments_October_combined.jsonl
    base: 1,268,638  +  supplement: 63,808  ->  combined: 1,332,446  (dupes removed: 0)
  r_politics_comments_November_combined.jsonl
    base: 426,419  +  supplement: 626,006  ->  combined: 1,052,425  (dupes removed: 0)
